# 📈 01. Initial Exploratory Data Analysis (EDA) - Kurs USD/IDR
**Mata Kuliah**: Machine Learning Operations (MLOps) TIF-B 2026  
**Mahasiswa**: Tobias Andra Valentino (NIM: 245150200111076)  
**Dosen Pengampu**: Rizal Setya Perdana, S.Kom., M.Kom., Ph.D.  
**Domain**: Financial & Quantitative Economics (Foreign Exchange Forecasting)  

---
### 🎯 Tujuan Eksperimen Awal:
1. Menganalisis karakteristik deret waktu kurs transaksi USD/IDR resmi Bank Indonesia.
2. Mengidentifikasi volatilitas, spread harian, dan autokorelasi (lag dependencies).
3. Menguji efektivitas benchmark industri *Naive Random Walk* (Efficient Market Hypothesis) sebagai baseline utama.
4. Merancang fitur prediktif bebas kebocoran data (*zero lookahead bias*) untuk model machine learning.

In [1]:
import sys
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Libraries successfully imported. Environment ready.")

## 1. Pemuatan Data Mentah (Raw Ingestion)
Data diambil dari database SQLite lokal `data/exchange_rates.db` yang bersumber dari API Web Service Bank Indonesia.

In [2]:
DB_PATH = Path("../data/exchange_rates.db") if Path("../data/exchange_rates.db").exists() else Path("data/exchange_rates.db")

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query("SELECT tgl, beli, jual, tengah, spread, mata_uang FROM exchange_rates ORDER BY tgl ASC", conn)

df["tgl"] = pd.to_datetime(df["tgl"])
print(f"Total data loaded: {len(df)} records")
print(f"Rentang Waktu: {df['tgl'].min()} hingga {df['tgl'].max()}")
df.head()

## 2. Pemeriksaan Kualitas Data & Ringkasan Statistik

In [3]:
print("=== Missing Values Check ===")
print(df.isnull().sum())

print("\n=== Ringkasan Statistik Kurs (IDR) ===")
df[["beli", "jual", "tengah", "spread"]].describe()

## 3. Visualisasi Tren Historis Kurs USD/IDR
Grafik di bawah menggambarkan pergerakan kurs jual, beli, dan kurs tengah sepanjang periode observasi.

In [4]:
plt.figure(figsize=(14, 6))
plt.plot(df["tgl"], df["jual"], label="Kurs Jual (Ask)", color="#e74c3c", lw=2)
plt.plot(df["tgl"], df["beli"], label="Kurs Beli (Bid)", color="#2ecc71", lw=2)
plt.plot(df["tgl"], df["tengah"], label="Kurs Tengah", color="#3498db", lw=1.5, linestyle="--")
plt.title("Dinamika Kurs Transaksi USD/IDR - Bank Indonesia", fontsize=14, fontweight="bold")
plt.xlabel("Tanggal Transaksi")
plt.ylabel("Nilai Kurs (Rupiah / USD)")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Analisis Spread & Volatilitas Harian

In [5]:
df["daily_return"] = df["jual"].pct_change() * 100
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df["daily_return"].dropna(), kde=True, ax=ax[0], color="#9b59b6")
ax[0].set_title("Distribusi Return Harian Kurs Jual (%)", fontweight="bold")
ax[0].set_xlabel("Daily Delta (%)")

ax[1].plot(df["tgl"], df["spread"], color="#e67e22", lw=1.8)
ax[1].set_title("Dinamika Spread (Jual - Beli) Harian (Rp)", fontweight="bold")
ax[1].set_xlabel("Tanggal Transaksi")
ax[1].set_ylabel("Spread (Rp)")

plt.tight_layout()
plt.show()

## 5. Autokorelasi & Analisis Lag
Mengevaluasi korelasi nilai kurs hari ini dengan nilai kurs pada beberapa hari bursa sebelumnya ($t-1$ s/d $t-14$).

In [6]:
lags = [1, 2, 3, 5, 7, 10, 14]
corrs = {f"Lag_{l}": df["jual"].autocorr(lag=l) for l in lags}

plt.figure(figsize=(9, 4))
sns.barplot(x=list(corrs.keys()), y=list(corrs.values()), palette="Blues_r")
plt.title("Autokorelasi Serial Kurs USD/IDR", fontsize=13, fontweight="bold")
plt.ylabel("Koefisien Korelasi (r)")
plt.ylim(0.8, 1.0)
for i, (k, v) in enumerate(corrs.items()):
    plt.text(i, v + 0.005, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Evaluasi Naive Random Walk Benchmark
Sesuai hipotesis pasar efisien (*Efficient Market Hypothesis*), estimasi terbaik hari esok adalah kurs hari ini ($y_{t+1} = y_t$). Model ML wajib diuji apakah mampu mengalahkan baseline ini.

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_true = df["jual"].iloc[1:].values
y_naive = df["jual"].iloc[:-1].values  # y_t predicts y_{t+1}

mae_rw = mean_absolute_error(y_true, y_naive)
rmse_rw = np.sqrt(mean_squared_error(y_true, y_naive))
mape_rw = np.mean(np.abs((y_true - y_naive) / y_true)) * 100.0

print("=== Naive Random Walk Benchmark Results ===")
print(f"MAE (Mean Absolute Error) : Rp {mae_rw:.2f}")
print(f"RMSE                     : Rp {rmse_rw:.2f}")
print(f"MAPE (Percentage Error)  : {mape_rw:.3f}%")
print(f"\nHurdle: Model ML yang dilatih pada pipeline MLOps harus memiliki MAE < Rp {mae_rw:.2f} untuk dinyatakan valid.")

## 7. Kesimpulan & Rekomendasi Pipeline MLOps
1. **Kualitas Data**: Data bersih tanpa nilai kosong (*zero missing values*), memiliki kontinuitas harian yang baik.
2. **Dependensi Temporal**: Autokorelasi Lag 1 mencapai **> 0.98**, mengindikasikan persistensi tren yang kuat.
3. **Benchmark Hurdle**: Standar industri Random Walk menghasilkan MAE yang sangat ketat (sekitar puluhan Rupiah). Model regresi L2 (Ridge) dan rolling momentum direkomendasikan pada `src/features/pipeline.py` untuk memitigasi fluktuasi mendadak.
4. **Langkah Berikutnya**: Integrasi pipeline feature engineering ke dalam GitHub Codespaces terstandar dan pelaksanaan deployment otomatis via GitHub Flow.